In [33]:
import numpy as np,sys
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


#from astropy.constants import h, c, k_B, m_p, R_sun, au
from astropy.io import fits
import astropy.units as u
import astropy.constants as const

import mcfacts
from mcfacts.vis import data
from mcfacts.vis import styles
from mcfacts.vis import plotting
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

from pagn import Sirko
from pagn import Thompson

In [34]:
def find_output(PATHNAME):
    """ Make df for output

    Parameters
    ----------
    pathname: string
        path name for desired file

    Returns
    -------
    df: DataFrame
        pandas DataFrame
    
     """
    df = pd.read_csv(PATHNAME,  names = ['gal','radius_rg', 'mass_final', 'chi_eff', 'spin_final', 'spin_angle_final', 'mass_1', 'mass_2', 
                  'spin_1', 'spin_2', 'spin_angle_1', 'spin_angle_2', 'gen_1', 'gen_2', 'time_merged', 'chi_p', 
                  'v_kick', 'lum_shock', 'lum_jet', 'id_num'], sep=" ", comment="#")
    return df

In [35]:
def si_from_r_g(smbh_mass, distance_rg):
    """Calculate the SI distance from r_g

    Parameters
    ----------
    smbh_mass : float
        Mass [M_sun] of the SMBH
    distance_rg : array_like
        Distances [r_{g,SMBH}]

    Returns
    -------
    distance : numpy.ndarray
        Distance in SI with :obj:`astropy.units.quantity.Quantity` type
    """
    # Calculate c and G in SI
    c = const.c.to('m/s')
    G = const.G.to('m^3/(kg s^2)')
    # Assign units to smbh mass
    if hasattr(smbh_mass, 'unit'):
        smbh_mass = smbh_mass.to('solMass')
    else:
        smbh_mass = smbh_mass * u.solMass
    # convert smbh mass to kg
    smbh_mass = smbh_mass.to('kg')
    # Calculate r_g in SI
    r_g = G*smbh_mass/(c ** 2)
    # Calculate distance
    distance = (distance_rg * r_g).to("m")

    assert np.isfinite(distance).all(), \
        "Finite check failure: distance"
    assert np.all(distance > 0).all(), \
        "distance contains values <= 0"

    return distance

In [36]:
disk_model = Sirko.SirkoAGN()
disk_model.solve_disk(N=1e4)

optical_depth = disk_model.tauV
density = disk_model.rho
radius = disk_model.R
height = disk_model.h
kappa = disk_model.kappa
teff = disk_model.Teff4**0.25
sound_speed = disk_model.cs

#disk_model.plot()
L_agn = (0.1 * 1.26) * 10**38 * (10**8 / 1) * u.erg / u.s
print(L_agn)

### Sirko & Goodman 2003 parameters ###
Mbh = 1.000000e+08 MSun
Mdot = 1.298344e+00 MSun/yr
le = 0.5
Rs = 9.570121e-06 pc
Rmin = 2.500000e+00 Rs
Rmax = 1.000000e+07 Rs, 9.570121e+01 pc
alpha = 0.01
b = 0
eps = 0.1
X = 0.7
Opacity = combined

debug = False
xtol = 1e-10
root method = lm
Q<1 at i=3408 (R=7.92e+02 Rs)
Beginning star formation at index 3408
Mdisk = 335459537.4383986 Msun
Mdisk/Mbh = 3.354595374383986
1.26e+45 erg / s


In [37]:
pagn_sg_disk = pd.DataFrame({'radius': radius,
                   'density': density,
                   'tau': optical_depth,
                   'height': height,
                   'kappa': kappa,
                   'cs': sound_speed})
pagn_sg_disk

,radius,density,tau,height,kappa,cs
0,1.773274e+12,4.005222e-07,2744.849523,1.989881e+11,0.034440,9.707911e+06
1,1.775816e+12,4.003327e-07,2747.158532,1.992486e+11,0.034440,9.699751e+06
2,1.778362e+12,4.001452e-07,2749.476565,1.995088e+11,0.034441,9.691573e+06
3,1.780912e+12,3.999599e-07,2751.803618,1.997690e+11,0.034441,9.683377e+06
4,1.783465e+12,3.997767e-07,2754.139687,2.000289e+11,0.034441,9.675163e+06
...,...,...,...,...,...,...
9995,2.934362e+18,1.252559e-18,0.000208,2.472285e+18,0.000067,5.666182e+04
9996,2.938569e+18,1.247187e-18,0.000207,2.477604e+18,0.000067,5.666183e+04
9997,2.942781e+18,1.241839e-18,0.000207,2.482934e+18,0.000067,5.666183e+04
9998,2.947000e+18,1.236513e-18,0.000206,2.488276e+18,0.000067,5.666184e+04


In [38]:
output_mergers_population = find_output("/Users/vprit/desktop/work/mcfacts/em_jets/output_mergers_population.dat")
# Ensure pandas shows all columns and allows wide output in the notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth', None)
# Keep numpy print width as before
np.set_printoptions(linewidth=2000)

output_mergers_population

,gal,radius_rg,mass_final,chi_eff,spin_final,spin_angle_final,mass_1,mass_2,spin_1,spin_2,spin_angle_1,spin_angle_2,gen_1,gen_2,time_merged,chi_p,v_kick,lum_shock,lum_jet,id_num
0,0.0,10456.200741,30.358008,-0.052439,0.688614,0.0,19.091799,12.930601,0.252755,0.247206,2.447393,0.882894,1.0,1.0,400000.0,0.161705,167.783144,2.767928e+41,4.455648e+43,207.0
1,0.0,5611.352706,25.521415,-0.015091,0.682028,0.0,13.586011,13.281736,0.092005,0.027418,1.924261,1.495966,1.0,1.0,430000.0,0.086317,53.311090,6.261938e+39,1.808514e+45,213.0
2,0.0,1349.641171,35.927171,-0.018391,0.664132,0.0,22.046777,15.693961,0.022439,0.125530,1.814120,1.866924,1.0,1.0,430000.0,0.081455,96.588424,8.864622e+39,3.083424e+46,218.0
3,0.0,13060.786722,42.894584,-0.095020,0.583053,0.0,12.403600,32.206929,0.062626,0.143770,1.225239,2.905591,1.0,1.0,440000.0,0.033616,199.659467,4.652656e+41,4.044154e+43,219.0
4,0.0,15141.498895,23.227126,-0.042550,0.669096,0.0,10.702451,13.693039,0.038529,0.069195,1.796762,3.079060,1.0,1.0,450000.0,0.028339,89.340596,5.812274e+40,3.701199e+43,217.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1427,99.0,6057.143054,27.792524,0.196725,0.726693,0.0,11.409177,17.924131,0.252496,0.215022,0.664886,0.429850,1.0,1.0,470000.0,0.133763,104.532292,4.794743e+40,5.185577e+44,223.0
1428,99.0,4272.651018,30.837613,0.440530,0.711038,0.0,12.376829,20.267641,0.019404,0.698236,0.302234,0.000000,1.0,2.0,470000.0,0.015608,93.870708,2.473119e+40,2.357479e+45,225.0
1429,99.0,5190.308469,32.421540,0.497266,0.640378,0.0,23.753158,10.195866,0.698538,0.067239,0.000000,1.135270,2.0,1.0,470000.0,0.023343,109.241266,4.484688e+40,1.318380e+45,229.0
1430,99.0,8162.682018,28.489525,-0.009358,0.638206,0.0,10.500188,19.346140,0.014391,0.104946,1.881377,1.685870,1.0,1.0,470000.0,0.104252,232.792657,6.459971e+41,4.027626e+43,233.0


In [39]:
radius_rg = output_mergers_population['radius_rg'].values
output_mergers_population['radius'] = np.array(si_from_r_g(1.000000e+08, radius_rg))
#output_mergers_population['radius']

In [40]:
# Ensure pandas shows all columns and allows wide output in the notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_colwidth', None)

# Sort by jet luminosity and show the sorted DataFrame
lum_sorted = output_mergers_population.sort_values('lum_jet')
# Keep numpy print width large for array printing
np.set_printoptions(linewidth=20000000000000)

lum_sorted

,gal,radius_rg,mass_final,chi_eff,spin_final,spin_angle_final,mass_1,mass_2,spin_1,spin_2,spin_angle_1,spin_angle_2,gen_1,gen_2,time_merged,chi_p,v_kick,lum_shock,lum_jet,id_num,radius
773,55.0,37486.006222,44.033597,-0.057666,0.580123,0.0,29.711465,16.196000,0.231521,0.232394,1.427882,2.871126,1.0,1.0,460000.0,0.229160,446.702242,9.543935e+42,1.871564e+41,224.0,5.535278e+15
17,1.0,18543.053080,23.947092,-0.160512,0.640714,0.0,12.842687,12.210652,0.343867,0.122602,2.195990,2.856849,1.0,1.0,490000.0,0.278824,423.025364,6.849021e+42,4.396246e+41,220.0,2.738114e+15
775,55.0,31304.297253,30.459390,-0.044834,0.610837,0.0,20.421666,11.447237,0.026566,0.261459,1.039995,2.176263,1.0,1.0,470000.0,0.111186,259.389851,1.933728e+42,6.597454e+41,216.0,4.622471e+15
1254,86.0,16565.013714,31.219242,-0.171407,0.626704,0.0,14.130305,18.577910,0.150389,0.300691,2.968910,2.250891,1.0,1.0,400000.0,0.205724,496.863113,9.410097e+42,6.837850e+41,216.0,2.446031e+15
1232,84.0,13132.566836,56.445209,0.289808,0.834564,0.0,37.140097,22.762170,0.225989,0.691460,1.376438,0.000000,1.0,2.0,420000.0,0.223238,772.927271,2.409501e+43,6.839977e+41,218.0,1.939188e+15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1007,68.0,1571.949196,95.118179,0.663809,0.870937,0.0,63.550075,37.856571,0.649830,0.687275,0.000000,0.000000,3.0,2.0,480000.0,0.000000,89.740186,2.936965e+39,1.860864e+47,245.0,2.321180e+14
975,68.0,977.345181,77.075915,0.562180,0.323832,0.0,10.533256,68.533368,0.090766,0.660332,2.571952,0.000000,1.0,2.0,200000.0,0.006096,68.258556,2.003170e+39,2.873772e+47,205.0,1.443172e+14
487,33.0,1051.101996,76.054478,0.682894,0.359289,0.0,10.333436,67.701287,0.081775,0.792188,1.988399,0.000000,1.0,3.0,360000.0,0.009240,64.145336,1.469765e+39,3.262391e+47,219.0,1.552084e+14
890,62.0,1076.754061,94.166881,0.741758,0.400323,0.0,82.244735,14.301740,0.873105,0.071931,0.000000,1.760686,4.0,1.0,430000.0,0.010039,70.461515,1.239226e+39,4.179042e+47,240.0,1.589962e+14


In [41]:
output_mergers_population = output_mergers_population.sort_values('radius')
output_mergers_population = pd.merge_asof(output_mergers_population, pagn_sg_disk, on='radius', suffixes = (None, '_pagn'))
output_mergers_population = output_mergers_population.sort_values('lum_jet')
print(output_mergers_population.shape)
print(output_mergers_population.iloc[905:910])

(1432, 26)
      gal    radius_rg  mass_final   chi_eff  spin_final  spin_angle_final     mass_1     mass_2    spin_1    spin_2  spin_angle_1  spin_angle_2  gen_1  gen_2  time_merged     chi_p      v_kick     lum_shock       lum_jet  id_num        radius       density           tau        height     kappa            cs
679  20.0  8158.492846   42.786095  0.495002    0.549778               0.0  10.637313  33.983514  0.050025  0.650882      1.630744      0.000000    1.0    2.0     420000.0  0.013456  102.863844  4.624120e+40  9.069571e+44   214.0  1.204703e+15  1.816861e-08  19412.199091  2.036002e+13  0.052478  56199.430637
466   6.0  4883.790036   45.908721 -0.128321    0.555091               0.0  35.552951  12.207720  0.197157  0.034104      2.688183      1.143460    1.0    1.0     370000.0  0.038017  205.427550  2.302383e+41  9.071388e+44   212.0  7.211527e+14  8.463465e-08  60336.408921  9.409923e+12  0.075761  56059.985748
408  88.0  4146.796375   24.360605  0.047053    0.696923   

In [42]:

jet_opening_angle = 10 * (np.pi / 180)  # Convert degrees to radians

output_mergers_population["lum_strength"] = (output_mergers_population["lum_jet"] / (np.pi * (jet_opening_angle* output_mergers_population["height"])**2 * output_mergers_population["density"]* const.c.cgs.value**3))

output_mergers_population["velocity"] = (
    (output_mergers_population["lum_strength"] **0.5)/
    (1+output_mergers_population["lum_strength"]**0.5)
)* const.c.cgs.value

output_mergers_population["t_bo"] = output_mergers_population["height"] / output_mergers_population["velocity"]

L_agn = 1.26e45 # erg / s
T_jet = 1e7 # s

# make a subset of mergers with remnants with a jet brighter than the agn:
brighter_than_agn = output_mergers_population[output_mergers_population['lum_jet'] > L_agn]

# get the subset of "bright" jets that will breakout before the jet turns off:
observable = brighter_than_agn[brighter_than_agn['t_bo'] < T_jet]

In [52]:
# Create a luminosity-sorted view of the merged DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

# Sort by jet luminosity (descending -> brightest first)
lum_sorted = output_mergers_population.sort_values('lum_jet', ascending=True)

print(lum_sorted.iloc[400:410])


       gal     radius_rg  mass_final   chi_eff  spin_final  spin_angle_final     mass_1     mass_2    spin_1    spin_2  spin_angle_1  spin_angle_2  gen_1  gen_2  time_merged     chi_p      v_kick     lum_shock       lum_jet  id_num        radius       density           tau        height     kappa            cs  lum_strength      velocity          t_bo
1157  43.0  15373.567330   22.304519  0.000493    0.679713               0.0  12.811795  10.644220  0.024036  0.040849      0.541184      2.190078    1.0    1.0     300000.0  0.026915   50.475899  1.073783e+40  7.105092e+43   208.0  2.270099e+15  2.706686e-09   6118.662385  5.287088e+13  0.042756  56328.518930      0.000004  5.710296e+07  9.258869e+05
1003  12.0  12653.335607   23.261433 -0.004217    0.681835               0.0  10.971943  13.484037  0.098347  0.057557      1.438472      1.892646    1.0    1.0     380000.0  0.077031   82.187805  4.070908e+40  7.111189e+43   221.0  1.868423e+15  4.856120e-09   9062.400614  3.944741e+13  0.0

In [47]:

lum_jet = lum_sorted.iloc[400]['lum_jet']
jet_opening_angle = 10 * (np.pi / 180)  # Convert degrees to radians
disk_height = lum_sorted.iloc[400]['height'] * 100 # Convert from m to cm
disk_dens = lum_sorted.iloc[400]['density'] * 1000 # Convert from kg/m^3 to g/cm^3

print(lum_jet,jet_opening_angle, disk_height, disk_dens)

7.105091653075504e+43 0.17453292519943295 5287088298781914.0 2.7066864292176647e-06


In [45]:
ratio = lum_jet/1.26e+45 

print("Luminosity Ratio (LSST u to AGN):", ratio)

Luminosity Ratio (LSST u to AGN): 0.7198072073396661
